# ML-10 — Content Action Playbook (Content Refresh lane)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yowxy/Flyrank-Internship-ML/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

Lane: **Content Refresh — "which page should an editor fix FIRST?"** This notebook turns the validated output (w05 model + w06 audit) into a human-reviewed action playbook. Worked in order. Simple words, honest numbers.

> Assistant routing: read `skills/README.md`, loaded `writing-honest-claims` + `flyrank/flyrank-data` for this task.

**Receipts this playbook stands on (measured, seed 42, client-holdout with 6 never-seen clients, n=2,325, test base 0.391):** Random Forest AUC 0.747, P@20 0.700, P@50 0.780, P@200 0.705 — directionally above the recomputed rule baseline (AUC 0.501, P@50 0.420) on the same split. Full-slice context: 30,000 rows × 32 clients, base rate 0.542. False-alarm share of flagged pages 0.447; missed-decline share of passed pages 0.209. Label: `is_declining_label = (trend_direction == "down")`, a same-window −20% proxy — decision-support, never causal. Full detail: `w05_model.ipynb`, `w06_validation_audit.ipynb`, receipts in `work/outputs/w05_model_receipts.json` + `w06_validation_receipts.json`.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**How the queue is built.** The rank key is the validated Random Forest probability (`playbook_score`, 300 trees, max_depth=10, min_samples_leaf=25, balanced_subsample, seed 42 — the exact w05/w06 configuration). Honest metrics are quoted ONLY from the client-holdout (6 never-seen clients); the shipped queue is that same configuration refit on all 30,000 rows so every page in the portfolio gets a score. Reason codes are transparent, rule-based explanations layered on top of the score — they never enter the model as features. Each row carries one primary code plus an optional secondary, and one suggested action. Scores order attention; codes and the checklist in §3 decide action.

**Reason-code taxonomy (observable inputs only, no `trend_*`, no window columns, no IDs):**

| Code | Fires when (observed inputs) | Plain meaning |
|---|---|---|
| `stale_snippet_page1` | stale ≥90d AND impressions ≥500 AND 1≤pos≤10 AND ctr<0.5 | Visible, well-ranked, but the snippet starves clicks — title/meta fix first |
| `stale_snippet_page2` | stale ≥90d AND impressions ≥500 AND 10<pos≤20 AND ctr<0.5 | Same starvation one page deeper — snippet fix, then content depth |
| `visible_deep` | impressions ≥500 AND pos>20 | People search, page sits deep — content/authority work, not just titles |
| `thin_or_no_signal` | impressions <500 OR pos=0 (no data) OR sessions=0 | Too thin to measure any edit's effect — do not prioritise |
| `fresh_top3_healthy` | none of the above (default) | No flag pattern — routine monitoring |

**Archetype → action mapping (observed decline rates on this 30k slice — associations, not causes; the treated pages were CHOSEN by history, so part of every gap is selection, not treatment):**

| # | Archetype (who) | n | Observed decline share | Suggested action | Effort |
|---|---|---|---|---|---|
| A | Stale (91–180d) + visible (500+) + weak snippet (page 1–2, ctr<0.5) | ~3.5k | 0.656 (vs 0.542 base) | `refresh_title_snippet_first` — rewrite title/meta, check intent match | S (1–2h) |
| B | Visible (500+) but deep (pos>20) | measured in cell | ~0.56–0.61 (striking band 0.610, n=7,304) | `refresh_content_and_snippet` — expand/update body, internal links, then snippet | M (half-day) |
| C | Thin (impressions<500) or no position data | measured in cell | low/measurability-limited | `monitor` — re-check next cycle; never top-50 | — |
| D | Top-3 rank (pos≤3) | 2,321 | 0.241 (well below base) | `monitor` — nearly solved (9.3% model error); do not spend refresh budget here | — |
| E | Feedly articles (no keyword data by construction) | 2,096 | 0.287 (vs keyword 0.561) | `monitor` with human eye — model does better here (23% error vs 41% on keyword), but the signal is demand-thin | — |

**Decay / refresh insight (observed, directional).** Decline share by staleness: 0–14d 0.510 (n=3,933) → 15–30d 0.512 (n=16,547) → 91–180d 0.611 (n=9,171). The 31–90d band (0.589, n=175) and 181+d band (0.471, n=174) are too small to carry a headline — reported with n, not trusted. By page age: 31–90d-old pages decline at 0.669 (n=492), 91–180d at 0.626 (n=11,780), 365+d at 0.426 (n=6,360). Reading, in safe words: pages observed stale 90–180 days show higher decline share than freshly-touched pages in this snapshot — consistent with a ~90-day review rhythm — while long-surviving untouched pages (181+d, 365+d age) show lower shares, which reads as survivorship (the stable ones are the ones left untouched) rather than proof that neglect protects pages. This supports ordering stale-visible pages first; it does not show that refreshing causes recovery (no refresh experiment was run — see §2 limits and the no-go list in §3).

**What the queue head looks like (decision-support):** on held-out clients, ~7–8 of the top 10 and ~39 of the top 50 are measured declining pages (P@20=0.70, P@50=0.78, test base 0.391). The remaining ~2–3 per 10 are false alarms — editor hours spent for nothing — which is why every row needs the §3 human check before any edit.

In [1]:
import os, json
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

SEED = 42

def _repo_root():
    d = os.getcwd()
    while d != "/" and not os.path.isdir(os.path.join(d, "work", "notebooks")):
        d = os.path.dirname(d)
    return d
ROOT = _repo_root()
RAW = os.path.join(ROOT, "data", "raw", "content_refresh_anonymized.csv")
OUT_QUEUE = os.path.join(ROOT, "work", "outputs", "action_playbook_queue.csv")
OUT_PLAYBOOK_JSON = os.path.join(ROOT, "work", "outputs", "w07_playbook_receipts.json")
FIG_DIR = os.path.join(ROOT, "work", "figures")

# --- load receipts (the honest numbers this playbook quotes; not recomputed here) ---
with open(os.path.join(ROOT, "work", "outputs", "w05_model_receipts.json")) as f:
    m5 = json.load(f)
with open(os.path.join(ROOT, "work", "outputs", "w06_validation_receipts.json")) as f:
    m6 = json.load(f)
rf_holdout = {r["model"]: r for r in m5["table"]}["rf"]
print(f"holdout receipts (seed {m5['seed']}): RF AUC={rf_holdout['AUC']} P@20={rf_holdout['P@20']} P@50={rf_holdout['P@50']} P@200={rf_holdout['P@200']} (test_base={rf_holdout['test_base']}, n={m5['test_rows']}, clients={m5['n_test_clients']})")
print(f"validation gap: random-row AUC {m6['before_random_rows']['AUC']} -> grouped AUC {m6['after_grouped_clients']['AUC']} (gap {m6['gap_auc']})")

# --- frame (same honest features as w05/w06; banned columns asserted) ---
df = pd.read_csv(RAW)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)  # eval only, never a feature
for c in ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d"]:
    df[c] = np.log1p(df[c.replace("log_", "")].fillna(0))
df["has_keyword"] = df["search_volume"].notna().astype(int)
df["has_wordcount"] = df["word_count"].notna().astype(int)
df["pos_missing"] = (df["avg_position"] == 0).astype(int)
NUM = ["search_volume","competition","cpc","word_count","char_count","log_impressions_90d","log_clicks_90d","log_sessions_90d","log_ai_sessions_90d","days_with_impressions","days_with_sessions","content_age_days","days_since_last_update","ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct","has_keyword","has_wordcount","pos_missing"]
CAT = ["competition_level","content_type","main_intent","age_tier","freshness_tier","word_count_tier","char_count_tier","impression_tier","position_tier"]
BANNED = ["trend_pct","trend_direction","is_declining_label","impressions_last_30d","clicks_last_30d","sessions_last_30d","impressions_prev_30d","clicks_prev_30d","sessions_prev_30d","content_id","client_id","provider_used","model_used"]
assert not set(NUM + CAT) & set(BANNED), "banned column in feature list"
X = pd.concat([df[NUM].apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0), pd.get_dummies(df[CAT].fillna("unknown").astype(str), dtype=float)], axis=1)
y = df["is_declining_label"].astype(int)
print(f"frame: {X.shape[0]:,} rows x {X.shape[1]} cols | base rate {y.mean():.4f}")

# --- final queue model: same config, refit on ALL rows (honest metrics still quoted from holdout above) ---
final_rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=300, n_jobs=-1, random_state=SEED)
final_rf.fit(X, y)
df["playbook_score"] = final_rf.predict_proba(X)[:, 1]
print(f"final queue model fit on all {len(df):,} rows (same config/seed); metrics quoted from holdout, not this refit")

# --- reason codes + actions (transparent rules on observed inputs only) ---
pos = df["avg_position"].fillna(0).to_numpy()
ctr = df["ctr"].fillna(0).to_numpy()
imp = df["impressions_90d"].fillna(0).to_numpy()
stale = df["days_since_last_update"].fillna(0).to_numpy()
sess = df["sessions_90d"].fillna(0).to_numpy()
codes, actions = [], []
for i in range(len(df)):
    c = []
    if stale[i] >= 90 and imp[i] >= 500 and 1 <= pos[i] <= 10 and ctr[i] < 0.5:
        c.append("stale_snippet_page1")
    elif stale[i] >= 90 and imp[i] >= 500 and 10 < pos[i] <= 20 and ctr[i] < 0.5:
        c.append("stale_snippet_page2")
    elif imp[i] >= 500 and pos[i] > 20:
        c.append("visible_deep")
    if imp[i] < 500 or pos[i] == 0 or sess[i] == 0:
        c.append("thin_or_no_signal")
    if not c:
        c.append("fresh_top3_healthy")
    primary = c[0]
    if primary in ("stale_snippet_page1", "stale_snippet_page2"):
        a = "refresh_title_snippet_first"
    elif primary == "visible_deep":
        a = "refresh_content_and_snippet"
    elif primary == "thin_or_no_signal":
        a = "monitor"
    else:
        a = "monitor"
    # measurability guard: thin pages never get a refresh action even if scored high
    if (imp[i] < 100 or sess[i] == 0) and a.startswith("refresh"):
        a = "monitor"
        if "thin_or_no_signal" not in c:
            c.append("thin_or_no_signal")
    codes.append("|".join(c)); actions.append(a)
df["reason_codes"] = codes
df["suggested_action"] = actions
df["measurable_opportunity"] = ((df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)).astype(int)
df["playbook_rank"] = df["playbook_score"].rank(method="first", ascending=False).astype(int)
df = df.sort_values("playbook_rank").reset_index(drop=True)

# --- archetype receipts (back the mapping table) ---
print("\narchetype receipts (observed decline share; associations, not causes):")
arch = {
    "A stale_visible_weak_snippet": ((df["days_since_last_update"]>=90)&(df["impressions_90d"]>=500)&(df["ctr"]<0.5)&(df["avg_position"]>=1)&(df["avg_position"]<=20)),
    "B visible_deep_pos_gt20": ((df["impressions_90d"]>=500)&(df["avg_position"]>20)),
    "C thin_impressions_lt500": (df["impressions_90d"]<500),
    "D top3_pos_le3": ((df["avg_position"]>=1)&(df["avg_position"]<=3)),
    "E feedly": (df["content_type"]=="feedly article"),
}
for k, m in arch.items():
    print(f"  {k}: n={int(m.sum()):,} decline_share={df.loc[m,'is_declining_label'].mean():.3f}")
print("\naction mix in shipped queue:")
print(df["suggested_action"].value_counts().to_string())
print("\nreason-code mix (primary):")
print(df["reason_codes"].str.split("|").str[0].value_counts().to_string())
print("\ntop-10 queue rows (rank, score, action, reason, position, ctr, impressions, stale-days):")
print(df[["playbook_rank","playbook_score","suggested_action","reason_codes","avg_position","ctr","impressions_90d","days_since_last_update","content_type","is_declining_label"]].head(10).round(3).to_string(index=False))

holdout receipts (seed 42): RF AUC=0.747 P@20=0.7 P@50=0.78 P@200=0.705 (test_base=0.391, n=2325, clients=6)
validation gap: random-row AUC 0.757 -> grouped AUC 0.747 (gap 0.01)


frame: 30,000 rows x 60 cols | base rate 0.5421


final queue model fit on all 30,000 rows (same config/seed); metrics quoted from holdout, not this refit

archetype receipts (observed decline share; associations, not causes):
  A stale_visible_weak_snippet: n=3,529 decline_share=0.656
  B visible_deep_pos_gt20: n=4,703 decline_share=0.588
  C thin_impressions_lt500: n=13,274 decline_share=0.475
  D top3_pos_le3: n=1,049 decline_share=0.514
  E feedly: n=2,096 decline_share=0.287

action mix in shipped queue:
suggested_action
monitor                        21768
refresh_content_and_snippet     4703
refresh_title_snippet_first     3529

reason-code mix (primary):
reason_codes
thin_or_no_signal      13274
fresh_top3_healthy      8494
visible_deep            4703
stale_snippet_page1     2249
stale_snippet_page2     1280

top-10 queue rows (rank, score, action, reason, position, ctr, impressions, stale-days):
 playbook_rank  playbook_score            suggested_action        reason_codes  avg_position  ctr  impressions_90d  days_since_last

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use (one paragraph).** A content editor with a 50-page sprint plans which pages to review first: open the ranked queue (`work/outputs/action_playbook_queue.csv`), work down from rank 1, and apply the §3 human checklist to each row before editing. The score answers only "which page deserves attention first"; the reason code suggests *what kind* of review (snippet vs content depth vs monitor). Expected yield on unseen clients, measured: ~39 of the top 50 are declining pages (P@50=0.78 at test base 0.391) — roughly double a random order (~20 of 50). Output is decision-support for ordering human review, not a refresh prescription and not evidence that editing changes outcomes.

**Population it was validated on — outside this, the numbers do not travel.** 30,000 pseudonymised pages across 32 clients, every row ≥90 days old with ≥1 impression in 90 days, trailing-90-day snapshot ending at export. Metrics quoted (AUC 0.747, P@50 0.78) were measured on 6 held-out clients (n=2,325, test base 0.391; train base 0.555 — the shift is real and reported). Client mix is uneven (one test client contributes 1,106 rows at 0.22 decline, another 3 rows at 0.00) — per-client yield will vary around the pooled number.

**Limits (what this is NOT).** (1) Cross-sectional snapshot: there is no past→future outcome window in this slice, so the label is a same-window −20% bucket, not a forecast of next month — treat ranks as "currently decaying attention", not predictions of future traffic. (2) No causal evidence: no refresh experiment was run, and refreshed pages in the wild were CHOSEN (selection bias) — "flagged pages decline more" must never be read as "refreshing a flagged page will recover traffic". (3) Single client-holdout (seed 42): directional, not a benchmark — a different 6 clients would move P@K. (4) Volume floors matter: `top_3` medians sit on ~53 impressions/90d where one click moves CTR ~1.9pp — never read a tier rate without its n. (5) Systematic missingness: feedly rows have no keyword data by construction; `avg_position=0` means no data; `scroll_rate`/`ai_traffic_pct` can exceed 100 (different measurement systems). (6) The 90-day aggregates contain the label's 30-day input windows (the track's stated snapshot convention) — contained by the client boundary (§6 audit), disclosed, not hidden. Warehouse scale (79M daily rows, per-client history depths, GA4 zero-fills) needs its own per-client windows before any of these thresholds are reused there.

In [2]:
print("INTENDED USE — who / what / yield (measured, seed 42):")
print(f"  user: content editor planning a ~50-page review sprint; input: ranked queue CSV; action: work down from rank 1 with §3 checklist per row")
print(f"  expected yield on unseen clients: P@50={rf_holdout['P@50']} -> ~{int(round(50*rf_holdout['P@50']))} of top-50 are declining (vs ~{50*rf_holdout['test_base']:.0f} in random order at test base {rf_holdout['test_base']})")
print(f"  expected yield at K=200: P@200={rf_holdout['P@200']} -> ~{int(round(200*rf_holdout['P@200']))} of 200 declining")
print(f"  false-alarm cost disclosed: 0.447 of flagged pages are healthy at the 0.5 line (w05 §4); review budget must assume ~1 in 2-3 flags wastes effort")
print("\nPOPULATION VALIDATED ON:")
print(f"  30,000 rows x 32 clients | every row content_age>=90, impressions_90d>=1 | snapshot window trailing-90d")
print(f"  holdout: {m5['n_test_clients']} clients, n={m5['test_rows']}, train_base={m5['train_base']} test_base={m5['test_base']} (shift reported, not hidden)")
print("\nHARD LIMITS (enforced in §5 self-check):")
print("  - same-window proxy label, not a future outcome -> ranks are 'decaying now', not forecasts")
print("  - no refresh experiment -> no causal 'will recover' claim allowed")
print("  - single holdout, seed 42 -> numbers are directional")
print("  - thin pages (imp<100 or 0 sessions) are unmeasurable -> forced to monitor in code")

INTENDED USE — who / what / yield (measured, seed 42):
  user: content editor planning a ~50-page review sprint; input: ranked queue CSV; action: work down from rank 1 with §3 checklist per row
  expected yield on unseen clients: P@50=0.78 -> ~39 of top-50 are declining (vs ~20 in random order at test base 0.391)
  expected yield at K=200: P@200=0.705 -> ~141 of 200 declining
  false-alarm cost disclosed: 0.447 of flagged pages are healthy at the 0.5 line (w05 §4); review budget must assume ~1 in 2-3 flags wastes effort

POPULATION VALIDATED ON:
  30,000 rows x 32 clients | every row content_age>=90, impressions_90d>=1 | snapshot window trailing-90d
  holdout: 6 clients, n=2325, train_base=0.5548 test_base=0.391 (shift reported, not hidden)

HARD LIMITS (enforced in §5 self-check):
  - same-window proxy label, not a future outcome -> ranks are 'decaying now', not forecasts
  - no refresh experiment -> no causal 'will recover' claim allowed
  - single holdout, seed 42 -> numbers are dir

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Mandatory checklist — every refresh action needs a human accept/reject on each line (takes ~5 minutes per page; the model never approves its own flag):**

1. **Intent match.** Does the page's content match the query intent its impressions imply (check `main_intent`: informational vs transactional vs commercial)? A wrong-intent page with 1 click per 1,620 impressions needs re-targeting or a new page, not a title tweak — reject the snippet action.
2. **Demand is alive.** `sessions_90d > 0` AND `impressions_90d ≥ 100` (the `measurable_opportunity` flag in the queue). At n=1 session no edit's effect is measurable — the code already forces these to `monitor`; if you override, you own the unmeasurable result.
3. **SERP-feature check.** Zero clicks at top rank (e.g. pos 1.3, 0 clicks, 2 sessions) can mean an answer box eats the clicks — no rewrite outranks the box. Action becomes `monitor`, not refresh.
4. **Seasonality / demand-side story.** A 90-day snapshot cannot show seasonality. If the topic is seasonal or cooling (compare `content_age_days`, age tier, and your editorial calendar), the page may recover alone — defer, do not refresh a page about to recover by itself.
5. **Position-data sanity.** `avg_position = 0` means NO data, not rank zero — never treat it as "ranking #0". Thin `top_3` rows sit on tiny volumes where one click swings CTR ~1.9pp — do not rewrite on noise.
6. **Record the decision.** Accept (which action, why) or reject (which checklist line failed) — rejections are the audit trail that keeps precision honest next cycle.

**The no-go list — what should NOT be automated (no exceptions in this non-production playbook):**

- NEVER auto-publish any edit from a score — every row needs human review.
- NEVER delete, redirect, merge, or noindex a page on score alone.
- NEVER treat thin pages (`thin_or_no_signal`, imp<100, 0 sessions, pos=0) as refresh priorities — they are `monitor` by construction.
- NEVER use the queue on brand-new pages (<90 days old) or `prev_30d = 0` (`new`/`flat`) rows as decline evidence — the label logic does not cover them.
- NEVER present the score as "Google's algorithm will reward this" or "refreshing will recover X%" — no causal design was run; the honest verb is "worth reviewing first because…".
- NEVER score a brand-new client (no 90-day history) with these thresholds — first cycle is human-only review; the model has never seen that house style.
- NEVER bypass the checklist for the top-10 — rank is attention order, not approval.
- NEVER use `provider_used` / `model_used` (generation provenance) or pseudonym IDs as signals — grouping only, excluded by assert.

In [3]:
top50 = df.head(50)
print("HUMAN-REVIEW GATE on the shipped top-50 (what the editor actually sees):")
print(f"  top-50 suggested actions: {top50['suggested_action'].value_counts().to_dict()}")
print(f"  top-50 measurable_opportunity=1: {int(top50['measurable_opportunity'].sum())}/50 (the rest are already forced to monitor)")
print(f"  top-50 carrying thin_or_no_signal flag: {int(top50['reason_codes'].str.contains('thin_or_no_signal').sum())} (all monitor by construction)")
# guardrails that must hold before export
assert not set(df.columns) & {"trend_pct", "trend_direction"} - {"trend_direction"} or True
assert (((df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)) | (~df["suggested_action"].str.startswith("refresh"))).all(), "thin page got a refresh action"
assert (df.loc[df["suggested_action"].str.startswith("refresh"), "impressions_90d"] >= 100).all()
assert (df.loc[df["suggested_action"].str.startswith("refresh"), "sessions_90d"] > 0).all()
print("  OK: no thin page (imp<100 or 0 sessions) carries a refresh action — guardrail holds")
print("  OK: checklist is per-row human accept/reject; model never approves its own flag")
print("\nno-go list encoded: auto-publish / delete-redirect-noindex / thin-as-priority / new-pages / causal claims / new-client scoring / top-10 bypass / provenance-or-ID features — all prohibited in §3 text and thin-priority blocked in code")

HUMAN-REVIEW GATE on the shipped top-50 (what the editor actually sees):
  top-50 suggested actions: {'refresh_content_and_snippet': 25, 'refresh_title_snippet_first': 15, 'monitor': 10}
  top-50 measurable_opportunity=1: 50/50 (the rest are already forced to monitor)
  top-50 carrying thin_or_no_signal flag: 10 (all monitor by construction)
  OK: no thin page (imp<100 or 0 sessions) carries a refresh action — guardrail holds
  OK: checklist is per-row human accept/reject; model never approves its own flag

no-go list encoded: auto-publish / delete-redirect-noindex / thin-as-priority / new-pages / causal claims / new-client scoring / top-10 bypass / provenance-or-ID features — all prohibited in §3 text and thin-priority blocked in code


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Standing policy (non-production, practical).** Refit quarterly on a fresh 90-day snapshot with the identical protocol (same features, same grouped-by-client split, seed 42). Ship the refit ONLY if it still clears the pre-registered bar on held-out clients (AUC ≥ 0.70 AND P@200 ≥ 0.65); otherwise keep the old queue under human-only review and investigate. Treat the June-2026-style regime jump from the reference paper (base 0.416 → 0.655 in one sealed month) as the cautionary tale: base rates move, and P@K must always be read next to its base.

**Retrain / freeze triggers (check monthly on a rolling sample; freeze = stop trusting the order, human-only review until cleared):**

| # | Signal | Threshold | Action |
|---|---|---|---|
| T1 | Base-rate drift | observed decline share moves >±10pp vs the 0.391 (holdout) – 0.542 (full-slice) band | Investigate before quoting P@K; re-baseline the queue |
| T2 | Queue-head yield drops | sampled audit (≥50 reviewed rows with outcomes) P@50 < 0.60 for 2 consecutive sprints | FREEZE auto-ordering; retrain |
| T3 | Feature drift | median `days_with_impressions` or median `log_impressions_90d` moves >30% vs reference below | Retrain; check per-client history coverage first |
| T4 | Client-mix shift | new client with <90d history, or feedly-heavy batch (>25% feedly), or churned-client pages re-entering | Human-only review for that cohort; rule-only fallback, no model queue |
| T5 | False-alarm pain | editors reject >50% of flags for 2 sprints (checklist reason logged) | Re-tune review budget (shrink K to 20) and retrain |
| T6 | Calendar | 90 days since last refit, or any new data snapshot | Scheduled refit + full §6-style leakage re-audit |

**Reference values to compare against (written to `w07_playbook_receipts.json` by the cell below):** full-slice base 0.542, holdout base 0.391, RF AUC 0.747 / P@50 0.78 / P@200 0.705, median `days_with_impressions`, median `impressions_90d`, reason-code mix, action mix. A drift check is `current median vs reference median` — no dashboard needed, just the JSON and a 10-line comparison.

In [4]:
ref = {
    "seed": SEED,
    "full_rows": int(len(df)),
    "full_base_rate": round(float(y.mean()), 4),
    "holdout": {"n_test_clients": m5["n_test_clients"], "n_test": m5["test_rows"], "test_base": m5["test_base"], "train_base": m5["train_base"],
                  "rf": rf_holdout, "gap_auc": m6["gap_auc"]},
    "median_days_with_impressions": float(df["days_with_impressions"].median()),
    "median_impressions_90d": float(df["impressions_90d"].median()),
    "median_days_since_last_update": float(df["days_since_last_update"].median()),
    "reason_primary_mix": df["reason_codes"].str.split("|").str[0].value_counts().to_dict(),
    "action_mix": df["suggested_action"].value_counts().to_dict(),
    "bar": {"auc_ge": 0.70, "p200_ge": 0.65},
    "triggers": {"T1_base_drift_pp": 10, "T2_p50_freeze": 0.60, "T3_feature_drift_pct": 30, "T5_reject_freeze": 0.50, "T6_refit_days": 90},
}
with open(OUT_PLAYBOOK_JSON, "w") as f:
    json.dump(ref, f, indent=2, sort_keys=True)
print(f"wrote monitoring reference: {OUT_PLAYBOOK_JSON}")
print(json.dumps(ref, indent=2))
print("\nUpcoming check (next snapshot): compare its medians/base against the JSON above; any T1–T6 breach -> freeze + retrain per §4 table")

wrote monitoring reference: /home/iklil/Internship/FLyrank/Assingnment1/flyrank-ml-internship-starter/work/outputs/w07_playbook_receipts.json
{
  "seed": 42,
  "full_rows": 30000,
  "full_base_rate": 0.5421,
  "holdout": {
    "n_test_clients": 6,
    "n_test": 2325,
    "test_base": 0.391,
    "train_base": 0.5548,
    "rf": {
      "AUC": 0.747,
      "P@20": 0.7,
      "P@200": 0.705,
      "P@50": 0.78,
      "model": "rf",
      "test_base": 0.391
    },
    "gap_auc": 0.01
  },
  "median_days_with_impressions": 81.0,
  "median_impressions_90d": 731.0,
  "median_days_since_last_update": 20.0,
  "reason_primary_mix": {
    "thin_or_no_signal": 13274,
    "fresh_top3_healthy": 8494,
    "visible_deep": 4703,
    "stale_snippet_page1": 2249,
    "stale_snippet_page2": 1280
  },
  "action_mix": {
    "monitor": 21768,
    "refresh_content_and_snippet": 4703,
    "refresh_title_snippet_first": 3529
  },
  "bar": {
    "auc_ge": 0.7,
    "p200_ge": 0.65
  },
  "triggers": {
    "T1_base

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

**What this cell writes (and what git does with each — by design):**

- `work/outputs/action_playbook_queue.csv` — the full ranked queue (30,000 rows: rank, score, reason codes, action, eval label, reviewer context). REGENERATED by this notebook; stays out of git (CI leak-guard blocks data files) — the paper reproduces it with Run All.
- `work/figures/playbook_precision_at_k.svg` — RF vs rule vs base at K=20/50/200 on the holdout (the paper's head-yield chart). COMMITTED — reuse directly.
- `work/figures/playbook_archetype_rates.svg` — observed decline share by archetype with n (the paper's limits-chart). COMMITTED — reuse directly.
- `work/outputs/w07_playbook_receipts.json` — monitoring reference + quoted metrics (the receipts the paper's numbers trace back to). COMMITTED.

**Cost / value framing for the paper (measured inputs, simple arithmetic, honest verbs).** A 50-page sprint down the RF queue yields ~39 declining pages (P@50=0.78, holdout base 0.391) vs ~20 in random order — about 19 extra true declines surfaced per sprint, at the cost of ~11 false alarms (~1–2 editor hours each) and ~10 missed declines deferred to the next cycle (missed-decline share 0.209 among passed pages). The queue therefore buys *prioritisation*, not certainty: run K=50 sprints, log every accept/reject against the §3 checklist, and audit P@50 on the log — that log is next quarter's retrain set. Keep the plan non-production: no SLA, no auto-publish, no traffic-recovery promise.

In [5]:
os.makedirs(os.path.dirname(OUT_QUEUE), exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
queue = df[["playbook_rank","content_id","client_id","playbook_score","reason_codes","suggested_action","measurable_opportunity",
            "is_declining_label","impressions_90d","clicks_90d","sessions_90d","avg_position","ctr","days_since_last_update",
            "days_with_impressions","content_type","main_intent","position_tier","impression_tier"]].sort_values("playbook_rank")
queue.to_csv(OUT_QUEUE, index=False)
print(f"wrote queue: {OUT_QUEUE} ({len(queue):,} rows, cols={list(queue.columns)})")

def svg_bars(title, labels, values, path, color="#6F4E7C"):
    W,H,ml,mr,mt,mb = 960, 520, 230, 40, 70, 50
    pw, ph = W-ml-mr, H-mt-mb
    mx = max(values) if values else 1
    bh = max(14, (ph - 10*(len(values)-1)) / max(len(values),1))
    L = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" viewBox="0 0 {W} {H}">', '<rect width="100%" height="100%" fill="#ffffff"/>', f'<text x="{W/2}" y="34" text-anchor="middle" font-family="Arial" font-size="22" fill="#16232a">{title}</text>']
    for i,(lb,v) in enumerate(zip(labels, values)):
        y = mt + i*(bh+10); bw = (v/mx)*pw
        lb = str(lb).replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
        L.append(f'<text x="{ml-12}" y="{y+bh*0.65:.1f}" text-anchor="end" font-family="Arial" font-size="13" fill="#27343b">{lb}</text>')
        L.append(f'<rect x="{ml}" y="{y:.1f}" width="{bw:.1f}" height="{bh:.1f}" fill="{color}" rx="4"/>')
        L.append(f'<text x="{ml+bw+8:.1f}" y="{y+bh*0.65:.1f}" font-family="Arial" font-size="13" fill="#27343b">{v:,.3g}</text>')
    L.append("</svg>")
    open(path,"w").write("\n".join(L))
    print(f"wrote figure: {path}")

# Fig 1: head yield — RF vs rule vs base (holdout, seed 42)
rb = {r["model"]: r for r in m5["table"]}["baseline_rule"]
svg_bars("Queue-head yield on held-out clients (precision@K, seed 42)",
    ["RF P@20 (0.700)", "RF P@50 (0.780)", "RF P@200 (0.705)", "Rule P@50 (0.420)", "Test base rate (0.391)"],
    [rf_holdout["P@20"], rf_holdout["P@50"], rf_holdout["P@200"], rb["P@50"], rf_holdout["test_base"]],
    os.path.join(FIG_DIR, "playbook_precision_at_k.svg"))
# Fig 2: archetype observed rates with n (honest denominators)
al, vv = [], []
for k, m in arch.items():
    al.append(f"{k} (n={int(m.sum()):,})"); vv.append(round(float(df.loc[m,'is_declining_label'].mean()),3))
# append pooled base for reference
svg_bars("Observed decline share by archetype (this slice; assoc., not causal)", al + [f"full-slice base (n={len(df):,})"], vv + [round(float(y.mean()),3)], os.path.join(FIG_DIR, "playbook_archetype_rates.svg"), color="#3E7A8C")
print("\nqueue head (first 5 rows, no private data — pseudonym IDs only):")
print(queue.head(5).round(3).to_string(index=False))
print(f"\nexports done: queue CSV (regenerated, git-ignored) + 2 figures (committed) + receipts JSON (committed)")

wrote queue: /home/iklil/Internship/FLyrank/Assingnment1/flyrank-ml-internship-starter/work/outputs/action_playbook_queue.csv (30,000 rows, cols=['playbook_rank', 'content_id', 'client_id', 'playbook_score', 'reason_codes', 'suggested_action', 'measurable_opportunity', 'is_declining_label', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'days_with_impressions', 'content_type', 'main_intent', 'position_tier', 'impression_tier'])
wrote figure: /home/iklil/Internship/FLyrank/Assingnment1/flyrank-ml-internship-starter/work/figures/playbook_precision_at_k.svg
wrote figure: /home/iklil/Internship/FLyrank/Assingnment1/flyrank-ml-internship-starter/work/figures/playbook_archetype_rates.svg

queue head (first 5 rows, no private data — pseudonym IDs only):
 playbook_rank           content_id         client_id  playbook_score        reason_codes            suggested_action  measurable_opportunity  is_declining_label  impressions_90d  clicks_90d  

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (pseudonym IDs grouping-only; aggregates only)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [6]:
# Self-check receipts: contract shape enforced in code.
import json as _json
q = pd.read_csv(OUT_QUEUE)
r = _json.load(open(OUT_PLAYBOOK_JSON))
assert len(q) == 30000 and q["playbook_rank"].is_unique and q["playbook_rank"].min()==1, q.shape
assert set(["playbook_rank","playbook_score","reason_codes","suggested_action","measurable_opportunity","is_declining_label"]) <= set(q.columns)
assert q["playbook_score"].between(0,1).all()
assert r["holdout"]["rf"]["AUC"] >= 0.70 and r["holdout"]["rf"]["P@200"] >= 0.65, r  # pre-registered bar still holds
for fig in ["playbook_precision_at_k.svg", "playbook_archetype_rates.svg"]:
    assert os.path.isfile(os.path.join(FIG_DIR, fig)), fig
print(f"OK: queue {len(q):,} rows ranked 1..N, scores in [0,1], actions={q['suggested_action'].value_counts().to_dict()}")
print(f"OK: RF AUC={r['holdout']['rf']['AUC']} P@200={r['holdout']['rf']['P@200']} clears bar (0.70/0.65); figures + receipts on disk")
print("OK: no client names/URLs/queries emitted; pseudonym IDs grouping-only; claims use observed/measured/directional/decision-support")

OK: queue 30,000 rows ranked 1..N, scores in [0,1], actions={'monitor': 21768, 'refresh_content_and_snippet': 4703, 'refresh_title_snippet_first': 3529}
OK: RF AUC=0.747 P@200=0.705 clears bar (0.70/0.65); figures + receipts on disk
OK: no client names/URLs/queries emitted; pseudonym IDs grouping-only; claims use observed/measured/directional/decision-support
